# Local Housing Database Playground
This script manages the lifecycle, custom querying, and data insertions of the local `housing` database.


### Step 0: Configuration Profile Mapping
Verify the paths and connection profile metrics below match your system configuration environment settings.

In [ ]:
import os
import subprocess
import random
import json

# Credentials mapping matching your custom installation folder hierarchy
PSQL_PATH = r"C:\Programme\PostgreSQL\18\bin\psql.exe"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "devi-group3"
DB_PASS = "fh-group3-admin"
DB_NAME = "housing"

# Pre-configure environment context variables to safely feed credentials without console lockups
os.environ["PGPASSWORD"] = DB_PASS

def run_sql(query, target_db="postgres"):
    """Helper routine executing SQL syntax statements directly via native subprocess channel pipelines."""
    cmd = [PSQL_PATH, "-h", DB_HOST, "-p", DB_PORT, "-U", DB_USER, "-d", target_db, "-c", query]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        return result.stdout.strip(), True
    except subprocess.CalledProcessError as e:
        return e.stderr.strip(), False

print(f"[+] Verification: Searching for native CLI engine client at: {PSQL_PATH}")
if os.path.exists(PSQL_PATH):
    print("[+] Ready! Native configuration setup successfully verified.")
else:
    print("[-] ERROR: psql.exe not found. Please verify your custom installation directory path setup mapping details.")

### 1) Create Database `housing`

In [ ]:
# Leverage master cluster template instance layer to register a new project workspace
check_query = f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}';"
output, success = run_sql(check_query, "postgres")

if success and "1" in output:
    print(f"[!] Database '{DB_NAME}' already exists.")
else:
    create_query = f"CREATE DATABASE {DB_NAME};"
    output, success = run_sql(create_query, "postgres")
    if success:
        print(f"[+] Database '{DB_NAME}' created successfully.")
    else:
        print(f"[-] Failed to generate targeted data namespace: {output}")

### 2) Create Table `housing_table` and Add 100 Random Rows

In [ ]:
create_table_sql = """
CREATE TABLE IF NOT EXISTS housing_table (
    id SERIAL PRIMARY KEY,
    median_house_value NUMERIC(12, 2),
    income NUMERIC(10, 2),
    house_age INT,
    rooms INT,
    bedrooms INT,
    population INT,
    households INT,
    lat NUMERIC(9, 6),
    lon NUMERIC(9, 6)
);
TRUNCATE TABLE housing_table;
"""

output, success = run_sql(create_table_sql, DB_NAME)
if not success:
    print(f"[-] Schema structural setup failed: {output}")
else:
    print("[+] Table structural mapping validated. Generating 100 simulation records natively...")
    
    # Build bulk transaction tracking script buffer block
    bulk_insert_script = "BEGIN;\n"
    for _ in range(100):
        income = round(random.uniform(20000, 150000), 2)
        house_age = random.randint(1, 60)
        rooms = random.randint(2, 10)
        bedrooms = random.randint(1, max(1, rooms - 1))
        population = random.randint(1, 6)
        households = random.randint(1, 3)
        median_house_value = round((income * 2.5) + (rooms * 12000) - (house_age * 500) + random.randint(-15000, 15000), 2)
        lat = round(random.uniform(47.0, 49.0), 6)
        lon = round(random.uniform(15.0, 17.0), 6)
        
        bulk_insert_script += f"""INSERT INTO housing_table (median_house_value, income, house_age, rooms, bedrooms, population, households, lat, lon) 
        VALUES ({median_house_value}, {income}, {house_age}, {rooms}, {bedrooms}, {population}, {households}, {lat}, {lon});\n"""
        
    bulk_insert_script += "COMMIT;"
    
    output, success = run_sql(bulk_insert_script, DB_NAME)
    if success:
        print("[+] 100 data rows successfully seeded and auto-committed to disk storage context!")
    else:
        print(f"[-] Seeding operation aborted: {output}")

### 3) Add New Custom Data and Save It

In [ ]:
def add_new_house(median_house_value, income, house_age, rooms, bedrooms, population, households, lat, lon):
    insert_sql = f"""
    INSERT INTO housing_table (median_house_value, income, house_age, rooms, bedrooms, population, households, lat, lon)
    VALUES ({median_house_value}, {income}, {house_age}, {rooms}, {bedrooms}, {population}, {households}, {lat}, {lon})
    RETURNING id;
    """
    output, success = run_sql(insert_sql, DB_NAME)
    if success:
        # Clean up output formatting to grab the generated primary index identifier token cleanly
        lines = [line.strip() for line in output.split('\n') if line.strip() and '-' not in line]
        new_id = lines[-1] if lines else "Unknown"
        print(f"[+] Successfully inserted and saved new entry! (Returned token lines: {new_id})")
    else:
        print(f"[-] Data record append failed: {output}")

print("Inserting manual record input...")
add_new_house(
    median_house_value=350000.00, 
    income=85000.50, 
    house_age=5, 
    rooms=5, 
    bedrooms=3, 
    population=4, 
    households=2, 
    lat=48.2082,
    lon=16.3738
)

### 4) Search Operations
This functional segment queries records leveraging custom filtering logic boundaries.

In [ ]:
def search_houses(min_age=None, max_age=None, min_rooms=None, max_rooms=None, min_income=None, max_income=None):
    query = "SELECT id, house_age, rooms, income, median_house_value FROM housing_table WHERE 1=1"
    
    if min_age is not None: query += f" AND house_age >= {min_age}"
    if max_age is not None: query += f" AND house_age <= {max_age}"
    if min_rooms is not None: query += f" AND rooms >= {min_rooms}"
    if max_rooms is not None: query += f" AND rooms <= {max_rooms}"
    if min_income is not None: query += f" AND income >= {min_income}"
    if max_income is not None: query += f" AND income <= {max_income}"
        
    query += " ORDER BY id DESC LIMIT 5;"
    
    output, success = run_sql(query, DB_NAME)
    if success:
        print(f"\n[🔍] Query execution completed. Matching response window data structure output:\n")
        print(output)
    else:
        print(f"[-] Filter query failed: {output}")

print("Testing search filters (using native psql tabular alignment text output formats):")
search_houses(min_age=1, max_age=30)

### 5) Delete Table `housing_table`

In [ ]:
drop_table_query = "DROP TABLE IF EXISTS housing_table;"
output, success = run_sql(drop_table_query, DB_NAME)
if success:
    print("[+] Table 'housing_table' successfully dropped from database registry.")
else:
    print(f"[-] Error while dropping table: {output}")